# PointNet × ModelNet40（Colab 本番実行）

ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から順に実行する。

条件は3つ（A: `pointnet` / A+T-Net: `pointnet_tnet` / B: `mlp_baseline`）。1条件あたり40分以内を想定した設定（100エポック）だが、**実測時間は未確認**なので、最初のエポックの `elapsed` を見て、長すぎる場合は `EPOCHS` を減らす。

In [ ]:
import torch
print(torch.__version__, 'cuda:', torch.cuda.is_available())
!nvidia-smi -L

## 1. リポジトリを clone して依存を入れる
`BRANCH` は実行したいブランチ（マージ済みなら `main`）。

In [ ]:
REPO = 'https://github.com/kimhuisong/summer-cv-practice.git'
BRANCH = 'main'  # 作業ブランチ上で試すときは 'claude/funny-meitner-97ygiy' に変える
!rm -rf summer-cv-practice && git clone --depth 1 -b {BRANCH} {REPO}
%cd summer-cv-practice/01_pointnet_modelnet40
!pip install -q h5py matplotlib scikit-learn

## 2. 動作確認（任意・数分）
`--quick` はCPU・データ一部・1エポック。**本番結果ではない**（`results/quick_*` に保存され、後でzipから除く）。

In [ ]:
!python train.py --quick --model pointnet

## 3. データ取得の確認
初回の学習時に `data/` へ自動ダウンロードされる（公式配布元の zip）。失敗する場合は README.md の「データの取得」を参照して手動で配置する。

In [ ]:
!python -c "from dataset import load_modelnet40; d = load_modelnet40('data'); print(d['source'], d['train_x'].shape, d['test_x'].shape)"

## 4. 本番学習（3条件）
各条件は `results/<条件名>/` に history.json・学習曲線・チェックポイント(`checkpoints/`)・metrics.json・混同行列・3D可視化を保存する。

In [ ]:
EPOCHS = 100
!python train.py --model pointnet --epochs {EPOCHS}

In [ ]:
!python train.py --model pointnet --use_tnet --epochs {EPOCHS}

In [ ]:
!python train.py --model mlp --epochs {EPOCHS}

## 5. 条件の比較（通常 vs シャッフル、入力点数）

In [ ]:
!python evaluate.py --compare
from IPython.display import Image, display
display(Image('results/comparison.png'))

## 6. results/ を zip にしてダウンロード
チェックポイント（`checkpoints/`）と動作確認用の `quick_*` は除く。

In [ ]:
!rm -f results.zip && zip -r results.zip results -x 'results/*/checkpoints/*' 'results/quick_*'
from google.colab import files
files.download('results.zip')